In [ ]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_295K_278464_17O_opt_magres_new.magres') #latest magres file from 2025

In [71]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [72]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [73]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [74]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-58.62788709 168.40679868 -80.11241991]
 [175.61811553  49.04275794  28.95345044]
 [  9.62496898 -25.67949218 -52.76157831]]

17O2 sigma:
 [[ -58.62788709 -168.40679868   80.11241991]
 [-175.61811553   49.04275794   28.95345044]
 [  -9.62496898  -25.67949218  -52.76157831]]

17O3 sigma:
 [[-58.62788709 168.40679868  80.11241991]
 [175.61811553  49.04275794 -28.95345044]
 [ -9.62496898  25.67949218 -52.76157831]]

17O4 sigma:
 [[ -58.62788709 -168.40679868  -80.11241991]
 [-175.61811553   49.04275794  -28.95345044]
 [   9.62496898   25.67949218  -52.76157831]]

17O5 sigma:
 [[ -47.44057414  228.4474648    55.076171  ]
 [ 197.49554429  102.00730098  -88.59347837]
 [  16.93245242  -50.39796985 -180.05968488]]

17O6 sigma:
 [[ -47.44057414 -228.4474648   -55.076171  ]
 [-197.49554429  102.00730098  -88.59347837]
 [ -16.93245242  -50.39796985 -180.05968488]]

17O7 sigma:
 [[ -47.44057414  228.4474648   -55.076171  ]
 [ 197.49554429  102.00730098   88.59347837]
 [ -16.93245242

In [75]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.352899935504245

17O2 sigma:
 6.35289993550426

17O3 sigma:
 6.35289993550424

17O4 sigma:
 6.352899935504253

17O5 sigma:
 8.335934183823582

17O6 sigma:
 8.335934183823564

17O7 sigma:
 8.335934183823579

17O8 sigma:
 8.33593418382361



In [76]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

atom_label = 0                                      # atom for which parameters are wanted
CS_total[:,:] = atoms.species('O').ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('O')[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = -0.0256 #electric quadrupole moment for O17 in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.875 -0.847  4.507]
 [-0.847 -0.448 -3.837]
 [ 4.507 -3.837 -0.427]]

CS Tensor:
 [[-58.628 168.407 -80.112]
 [175.618  49.043  28.953]
 [  9.625 -25.679 -52.762]]

CS isotropic Tensor:
 [[-20.782   0.      0.   ]
 [  0.    -20.782   0.   ]
 [  0.      0.    -20.782]]

CS symmetric Tensor:
 [[-58.628 172.012 -35.244]
 [172.012  49.043   1.637]
 [-35.244   1.637 -52.762]]

CS antisymmetric Tensor:
 [[  0.     -3.606 -44.869]
 [  3.606   0.     27.316]
 [ 44.869 -27.316   0.   ]]


In [77]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.35786484 -0.72623556 -5.63162929] 

 Unsorted Eigenvectors:
 [[-0.60852658  0.64908449 -0.45649177]
 [ 0.44584925  0.75554533  0.47996843]
 [-0.65644029 -0.08854704  0.74916324]] 

Sorted Eigenvalues: 
 [-0.72623556 -5.63162929  6.35786484] 

Sorted Eigenvectors: 
 [[ 0.64908449 -0.45649177 -0.60852658]
 [ 0.75554533  0.47996843  0.44584925]
 [-0.08854704  0.74916324 -0.65644029]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 177.12774072 -191.3058407   -48.16860749] 

 Unsorted Eigenvectors:
 [[ 0.59556897  0.79458741 -0.11801884]
 [ 0.79872851 -0.57009213  0.19242589]
 [-0.08561758  0.2088679   0.97418881]] 

Sorted Eigenvalues: 
 [ -48.16860749 -191.3058407   177.12774072] 

Sorted Eigenvectors: 
 [[-0.11801884  0.79458741  0.59556897]
 [ 0.19242589 -0.57009213  0.79872851]
 [ 0.97418881  0.2088679  -0.08561758]] 



In [78]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.7262355550201142 -5.63162928578278 6.357864840802952
CSA Tensor Components δyy, δxx, δzz: 
 -48.16860748877795 -191.30584069627116 177.12774072268076


In [79]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        6.35786
etaq            0.771547
iso_cs (ppm)  -20.7822
csa (ppm)     197.91
etas            0.723244


In [80]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.45649177  0.64908449 -0.60852658]
 [ 0.47996843  0.75554533  0.44584925]
 [ 0.74916324 -0.08854704 -0.65644029]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-6.7407773677975555 131.02895094254038 36.22911971513138 

Direction cosine csa: 

[[ 0.79458741 -0.11801884  0.59556897]
 [-0.57009213  0.19242589  0.79872851]
 [ 0.2088679   0.97418881 -0.08561758]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
77.89887994306643 94.91153910407567 -53.29008291580815 



In [81]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 28.84797239269381 chi: 87.13999440825735 xi: -87.28260868443421 



**Rotation of tensors Crystal--> Tenon Frame**

In [82]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[-58.62788709 172.0124571  -35.24372546]
 [172.0124571   49.04275794   1.63697913]
 [-35.24372546   1.63697913 -52.76157831]]
CSA Tensor in Tenon Frame: 
 [[  88.15168155 -101.32407161 -120.41392083]
 [-101.32407161  -59.55993084  -24.21062612]
 [-120.41392083  -24.21062612  -90.93845817]]
Quad Tensor in Crystal Frame: 
 [[ 0.87483019 -0.84721536  4.50740458]
 [-0.84721536 -0.44810118 -3.83718333]
 [ 4.50740458 -3.83718333 -0.42672901]]
Quad Tensor in Tenon Frame: 
 [[ 0.1593599  -1.3308563   3.14904105]
 [-1.3308563  -4.89081506  1.21607982]
 [ 3.14904105  1.21607982  4.73145517]]
